# Phase 4 · Pseudonymisation Walkthrough

Plain Phase 3 redaction collapses every named entity into a generic tag:

> *"Maria Petrova sued John Doe for breach of the Acme contract"*  becomes
> *"[PERSON] sued [PERSON] for breach of the [ORG] contract"*

A downstream LLM cannot answer "**who** sued **whom**?" from that — referential identity is gone.

**Pseudonymisation** preserves the structure by assigning each *distinct* entity a stable token:

> *"[PERSON_A] sued [PERSON_B] for breach of the [ORG_A] contract"*

The same surface form gets the same token within the document. The mapping (`vault`) is held by the firm and never leaves the firm's network. When the LLM's answer comes back referencing the tokens, the firm runs `restore()` locally to swap them back to real names.

This notebook walks through:
1. The `pseudonymise=True` flag on Lite and Pro pipelines.
2. How the substring-rule coreference handles the same person mentioned multiple ways ("Maria Petrova" / "Maria" / "Mrs Petrova").
3. The full round-trip: redact → query LLM (simulated) → restore.

---


## Setup


In [ ]:
import sys
sys.path.insert(0, "../../src")

import json
from anonymisation.pipeline import LitePipeline, ProPipeline, MosaicScorer, restore

# Hand-crafted predictor — same pattern as in the static showcase.
# Real production would use spaCy / the Phase 2 fine-tuned model.
def make_predictor(text, items):
    spans = []
    for needle, etype in items:
        idx = text.find(needle)
        if idx == -1:
            raise ValueError(f"{needle!r} not in text")
        spans.append((idx, idx + len(needle), etype, needle))
    def predict(_): return spans
    return predict


## Example 1 — referential identity in a contract dispute


In [ ]:
text = (
    "Maria Petrova sued Acme Holdings Ltd over the contract dated 12 March 2018. "
    "Mrs Petrova, who lives in Plovdiv, alleges Acme breached terms 3 and 7. "
    "Petrova is represented by Sofia District Court. Acme denies the claim."
)

entities = [
    ("Maria Petrova",        "PERSON"),
    ("Acme Holdings Ltd",    "ORG"),
    ("12 March 2018",        "DATETIME"),
    ("Mrs Petrova",          "PERSON"),    # same person as Maria Petrova
    ("Plovdiv",              "LOC"),
    ("Acme",                 "ORG"),       # same org as Acme Holdings Ltd
    ("Petrova",              "PERSON"),    # same person again
    ("Sofia District Court", "ORG"),
    ("Acme",                 "ORG"),       # same org again (last sentence)
]
predict = make_predictor(text, entities)

# Plain Lite — every PERSON becomes [PERSON], every ORG becomes [ORG]
plain = LitePipeline(ner_provider=predict)
plain_result = plain(text)

# Pseudonymised Lite — referential tokens
pseudo = LitePipeline(ner_provider=predict, pseudonymise=True)
pseudo_result = pseudo(text)

print("PLAIN:")
print("  " + plain_result.redacted_text)
print()
print("PSEUDONYMISED:")
print("  " + pseudo_result.redacted_text)


Notice three things in the pseudonymised version:

1. **Coreference works.** "Maria Petrova" → `[PERSON_A]`. "Mrs Petrova" → `[PERSON_A]`. "Petrova" → `[PERSON_A]`. All three forms collapse to the same token because each shorter form is a word-aligned substring of the longer one.
2. **Distinct entities get distinct tokens.** Maria gets PERSON_A; if there were a second person introduced later, they'd be PERSON_B.
3. **The vault is human-readable.** The firm holds it and uses it to round-trip LLM answers.


In [ ]:
print("VAULT:")
for token, original in pseudo_result.pseudonym_vault.items():
    print(f"  {token!r:18s} -> {original!r}")


## How the coreference rule works

The matcher uses a word-boundary substring rule:

- Two surface forms refer to the same entity if one appears, **as whole words**, inside the other.
- Same entity type only (so a LOC `Plovdiv` won't merge with a DEM `Plovdiv-resident`).

Limits worth knowing:

- "Smith" twice in the same document, referring to two different people, would incorrectly merge. A real coref model would handle this; the substring rule is a pragmatic baseline.
- Initialism / abbreviation gaps ("International Business Machines" / "IBM") won't merge unless one literally contains the other.
- Pronouns ("she", "he", "they") are out of scope — the NER model wouldn't tag them as PERSON in the first place.


In [ ]:
# Demonstrate the coref rule explicitly
from anonymisation.pipeline import Pseudonymiser

ps = Pseudonymiser()
print(ps.token_for("PERSON", "Maria Petrova"))   # [PERSON_A]  — new
print(ps.token_for("PERSON", "Maria"))           # [PERSON_A]  — substring
print(ps.token_for("PERSON", "Petrova"))         # [PERSON_A]  — substring
print(ps.token_for("PERSON", "Mrs Petrova"))     # [PERSON_A]  — substring
print(ps.token_for("PERSON", "John Smith"))      # [PERSON_B]  — new entity
print(ps.token_for("PERSON", "Smith"))           # [PERSON_B]  — substring of John Smith
print()
print("Vault:")
for k, v in ps.vault.items():
    print(f"  {k}: {v!r}")


## The round-trip — query an LLM, restore the answer

This is the workflow that makes pseudonymisation useful: you redact the document, send the redacted text to a model with a question, get back an answer that references `[PERSON_A]` / `[PERSON_B]`, then restore the answer to original names locally.

We simulate the LLM step rather than calling out to one. The point is to show the round-trip pattern, not to evaluate model quality.


In [ ]:
# 1. We've already redacted the text above. Pretend we ship pseudo_result.redacted_text
#    to a third-party LLM along with the question.

prompt = f"""You are a legal assistant. Read the redacted matter note and answer briefly.

DOCUMENT:
{pseudo_result.redacted_text}

QUESTION: Who is being sued, and where do they live?
"""

print("PROMPT THE LLM SEES (note: no real names):")
print(prompt)

# 2. The LLM responds (simulated — what a real model would plausibly say).
llm_answer = (
    "Based on the document, [ORG_A] is being sued. "
    "The plaintiff [PERSON_A] lives in [LOC]. "
    "The case is being handled by [ORG_B]."
)
print("\nLLM RAW ANSWER (still using tokens):")
print(llm_answer)

# 3. Restore locally, using only the vault we held onto.
restored_answer = restore(llm_answer, pseudo_result.pseudonym_vault)
print("\nRESTORED ANSWER (real names, local environment only):")
print(restored_answer)


The round-trip preserves three things the firm cares about:

- **Privacy.** The LLM saw `[PERSON_A]` and `[ORG_A]`, not "Maria Petrova" and "Acme Holdings Ltd". Even if the LLM provider logs every prompt, no client names land in their logs.
- **Utility.** The LLM could still reason about who-did-what to whom because the referential structure was preserved.
- **Locality of the secret.** The firm's mapping never leaves the firm. Even if the LLM is breached, the attacker has no Rosetta Stone — `[PERSON_A]` is just an opaque label without the vault.

Note that `[LOC]` came back unchanged in the restored answer — that's because Plovdiv was a QUASI in our example, not a DIRECT, so it wasn't pseudonymised. Pro mode would have generalised it instead. (Pseudonymisation is a DIRECT-only feature in this implementation.)


## Pro + pseudonymisation

Both flags compose. With `pseudonymise=True` on Pro, you get DIRECT pseudonymisation *plus* mosaic-aware QUASI generalisation.


In [ ]:
# Tiny synthetic haystack, just enough to make the iterate loop demonstrable
sigs = [
    (("DATETIME", "2018"), ("LOC", "bulgaria")),
] * 6
scorer = MosaicScorer(sigs)

pro = ProPipeline(
    ner_provider=predict,
    scorer=scorer,
    k_target=5,
    max_iterations=3,
    pseudonymise=True,
)
result = pro(text)

print("PRO + PSEUDONYMISED:")
print("  " + result.redacted_text)
print()
print(f"k_initial={result.mosaic_risk_initial}, k_final={result.mosaic_risk_final}, "
      f"iters={result.iterations_used}, converged={result.converged}")
print()
print("VAULT:")
for token, original in result.pseudonym_vault.items():
    print(f"  {token!r:18s} -> {original!r}")


## Caveats and future work

- **Per-document scope.** Each `Pseudonymiser` instance is independent. "Maria Petrova" in document A and "Maria Petrova" in document B get assigned independently — they may both be `[PERSON_A]` but the tokens are not portable. A persistent registry across documents is a future enhancement (and raises real key-management questions).
- **Vault is sensitive.** The vault is the secret. Encrypt at rest, audit access, treat it like a session key. The pipeline writes it to plain JSON because that's appropriate for a portfolio demo, not because it's appropriate for production.
- **Coref is heuristic.** The substring rule is fast and predictable but it will misfire on coincidentally-shared surnames or initialisms. Drop in a proper coref model for a deployment.
- **DIRECTs only.** QUASI generalisation in Pro stays plain (no `[LOC_A]` for individual cities) — the two features are conceptually orthogonal and combining them would conflate "this is a name we're hiding" with "this is a place we're broadening".

Static showcase has a fourth panel (`demo/index.html` ➝ Sample 4) demonstrating the full round-trip end-to-end, suitable for embedding in your portfolio site.
